In [3]:
import pandas as pd
import numpy as np


url = "https://docs.google.com/spreadsheets/d/1rFyU55Pu7nm9BZItX_qy-XF-gSfeSRgzejEchmZHnV4/export?format=csv"
data = pd.read_csv(url)


data.columns = data.columns.str.strip().str.replace(" ", "").str.replace(".", "")

print("Columns:", data.columns.tolist())
print("\nFirst 5 rows:")
print(data.head())


target_col = None
for col in data.columns:
    if data[col].nunique() == 2:
        target_col = col
        break

if target_col is None:
    raise ValueError("No binary target column found")

print("\nTarget column:", target_col)


def entropy(column):
    values, counts = np.unique(column, return_counts=True)
    probs = counts / counts.sum()
    return -np.sum(probs * np.log2(probs))


def information_gain(data, feature, target):
    total_entropy = entropy(data[target])
    values, counts = np.unique(data[feature], return_counts=True)

    weighted_entropy = 0
    for v, c in zip(values, counts):
        subset = data[data[feature] == v]
        weighted_entropy += (c / counts.sum()) * entropy(subset[target])

    return total_entropy - weighted_entropy


features = [col for col in data.columns if col != target_col]

ig_scores = {}
for feature in features:
    ig_scores[feature] = information_gain(data, feature, target_col)

print("\nInformation Gain for each feature:")
for k, v in ig_scores.items():
    print(f"{k}: {v}")

best_feature = max(ig_scores, key=ig_scores.get)
print("\nBest feature for decision tree:", best_feature)


decision_map = {}
for val in data[best_feature].unique():
    subset = data[data[best_feature] == val]
    decision_map[val] = subset[target_col].mode()[0]

print("\nDecision Tree Rule:")
for k, v in decision_map.items():
    print(f"If {best_feature} == {k} → Team {v}")


def decision_tree(row):
    return decision_map[row[best_feature]]

data["Predicted_Team"] = data.apply(decision_tree, axis=1)


accuracy = np.mean(data["Predicted_Team"] == data[target_col])
print("\nAccuracy:", accuracy)

print("\nSample Predictions:")
print(data[[best_feature, target_col, "Predicted_Team"]].head())




Columns: ['EmployeeID', 'Income', 'Gender', 'Department', 'Team']

First 5 rows:
   EmployeeID  Income Gender Department Team
0           1   46000      F    Finance    A
1           2   39000      M    Finance    B
2           3   41000      M   Software    A
3           4   33000      M   Software    B
4           5   30000      F         HR    A

Target column: Gender

Information Gain for each feature:
EmployeeID: 0.8112781244591328
Income: 0.8112781244591328
Department: 0.015712127384097774
Team: 0.31127812445913283

Best feature for decision tree: EmployeeID

Decision Tree Rule:
If EmployeeID == 1 → Team F
If EmployeeID == 2 → Team M
If EmployeeID == 3 → Team M
If EmployeeID == 4 → Team M
If EmployeeID == 5 → Team F
If EmployeeID == 6 → Team M
If EmployeeID == 7 → Team M
If EmployeeID == 8 → Team M
If EmployeeID == 9 → Team M
If EmployeeID == 10 → Team M
If EmployeeID == 11 → Team M
If EmployeeID == 12 → Team M
If EmployeeID == 13 → Team M
If EmployeeID == 14 → Team M
If Employee